# Notebook 09a — Exploração da estrutura do CalBased Test

**Projeto de Iniciação Científica PIBIC — PUC-SP**

Mini-notebook investigativo: antes de implementar a extração de features no subset CalBased (1,43 GB), é necessário entender:

1. **Quais chaves o arquivo `.mat` contém?**
2. **Como os pares (calibração, teste) estão estruturados?**
3. **Quantos segmentos no total? Quantos por sujeito?**
4. **A estrutura é compatível com o pipeline já desenvolvido?**

O objetivo é evitar implementar um notebook completo baseado em suposições erradas sobre o formato dos dados.

---


## 1. Carregamento

In [1]:
import os
from pathlib import Path
from datetime import datetime
import numpy as np
import mat73

BASE_IC = Path(r"C:\IC_PPG")
arquivo = (BASE_IC / "data" / "pulsedb_kaggle" / "datasets" /
           "weinanwangrutgers" / "pulsedb-balanced-training-and-testing" /
           "versions" / "4" / "VitalDB_CalBased_Test_Subset.mat")

print(f"Arquivo: {arquivo.name}")
print(f"Tamanho: {arquivo.stat().st_size / (1024**3):.2f} GB")
print(f"Carregando... (pode demorar 30-60s para arquivo de 1.4 GB)")

inicio = datetime.now()
dados = mat73.loadmat(str(arquivo))
duracao = (datetime.now() - inicio).total_seconds()
print(f"Carregado em {duracao:.1f} s")
print()
print(f"Chaves no nivel raiz: {list(dados.keys())}")

Arquivo: VitalDB_CalBased_Test_Subset.mat
Tamanho: 1.43 GB
Carregando... (pode demorar 30-60s para arquivo de 1.4 GB)
Carregado em 65.5 s

Chaves no nivel raiz: ['Subset']


## 2. Estrutura do Subset

In [2]:
subset = dados["Subset"]

print("Tipo de 'Subset':", type(subset))
if isinstance(subset, dict):
    print(f"\nChaves dentro de 'Subset' ({len(subset)} total):")
    print("-" * 60)
    for k in sorted(subset.keys()):
        v = subset[k]
        if hasattr(v, "shape"):
            print(f"  {k:<25} | shape={v.shape}, dtype={v.dtype}")
        elif isinstance(v, list):
            print(f"  {k:<25} | list, len={len(v)}, "
                  f"primeiro elem: {type(v[0]).__name__}")
        else:
            print(f"  {k:<25} | {type(v).__name__}: {v}")
else:
    print("Estrutura nao eh dict, ver atributos:", dir(subset))

Tipo de 'Subset': <class 'dict'>

Chaves dentro de 'Subset' (9 total):
------------------------------------------------------------
  Age                       | shape=(51720,), dtype=float64
  BMI                       | shape=(51720,), dtype=float64
  DBP                       | shape=(51720,), dtype=float64
  Gender                    | list, len=51720, primeiro elem: list
  Height                    | shape=(51720,), dtype=float64
  SBP                       | shape=(51720,), dtype=float64
  Signals                   | shape=(51720, 3, 1250), dtype=float64
  Subject                   | list, len=51720, primeiro elem: list
  Weight                    | shape=(51720,), dtype=float64


## 3. Inspeção detalhada de cada campo

In [3]:
# Tenta mostrar os primeiros valores de cada campo
print("PRIMEIROS VALORES DE CADA CAMPO")
print("=" * 70)
for k in sorted(subset.keys()):
    v = subset[k]
    print(f"\n--- {k} ---")
    if hasattr(v, "shape"):
        print(f"Shape: {v.shape}, dtype: {v.dtype}")
        if v.ndim == 1:
            print(f"Primeiros 5: {v[:5]}")
            if v.dtype.kind in 'fi':
                print(f"min={v.min()}, max={v.max()}, "
                      f"mean={v.mean():.2f}, std={v.std():.2f}")
        elif v.ndim == 2:
            print(f"Primeira linha: {v[0]}")
        elif v.ndim == 3:
            print(f"Shape: {v.shape}")
            print(f"Primeiro elemento (forma): {v[0].shape if hasattr(v[0], 'shape') else type(v[0])}")
    elif isinstance(v, list):
        print(f"Lista de {len(v)} elementos")
        print(f"Primeiros 3: {v[:3]}")
        # Se forem listas aninhadas, desempacota
        if len(v) > 0 and isinstance(v[0], list):
            print(f"Primeiro elemento desempacotado: {v[0]}")
            print(f"Tipo do primeiro: {type(v[0][0]) if len(v[0]) > 0 else 'vazio'}")

PRIMEIROS VALORES DE CADA CAMPO

--- Age ---
Shape: (51720,), dtype: float64
Primeiros 5: [48. 48. 48. 48. 48.]
min=0.4, max=92.0, mean=58.98, std=15.03

--- BMI ---
Shape: (51720,), dtype: float64
Primeiros 5: [20.7 20.7 20.7 20.7 20.7]
min=12.4, max=40.3, mean=22.92, std=3.44

--- DBP ---
Shape: (51720,), dtype: float64
Primeiros 5: [68.48212338 66.04714694 66.37822938 59.9107074  89.85084823]
min=22.46458172845342, max=240.99951599526537, mean=62.94, std=12.07

--- Gender ---
Lista de 51720 elementos
Primeiros 3: [['M'], ['M'], ['M']]
Primeiro elemento desempacotado: ['M']
Tipo do primeiro: <class 'str'>

--- Height ---
Shape: (51720,), dtype: float64
Primeiros 5: [177.4 177.4 177.4 177.4 177.4]
min=54.0, max=186.6, mean=162.50, std=9.64

--- SBP ---
Shape: (51720,), dtype: float64
Primeiros 5: [116.80487052 113.28881884 113.34018017  95.06269843 140.30255454]
min=46.55086164208538, max=248.48202055541915, mean=115.50, std=18.85

--- Signals ---
Shape: (51720, 3, 1250), dtype: float

## 4. Procura por campos de Cal vs Test

In [4]:
print("BUSCA POR CAMPOS DE CALIBRACAO E TESTE")
print("=" * 70)

# Procura por padroes 'Cal' e 'Test' nos nomes das chaves
chaves_cal  = [k for k in subset.keys() if "Cal"  in k or "cal"  in k]
chaves_test = [k for k in subset.keys() if "Test" in k or "test" in k]

print(f"\nChaves contendo 'Cal':  {chaves_cal}")
print(f"Chaves contendo 'Test': {chaves_test}")

# Se nao houver, talvez haja um campo Type/Split
chaves_split = [k for k in subset.keys() if k.lower() in
                ["type", "split", "phase", "stage", "subset", "set"]]
print(f"\nChaves de tipo/split: {chaves_split}")

BUSCA POR CAMPOS DE CALIBRACAO E TESTE

Chaves contendo 'Cal':  []
Chaves contendo 'Test': []

Chaves de tipo/split: []


## 5. Quantidade de sujeitos e segmentos

In [5]:
# Sujeitos
if "Subject" in subset:
    subjects = subset["Subject"]
    if isinstance(subjects, list):
        # Desempacota se for lista de listas
        if len(subjects) > 0 and isinstance(subjects[0], list):
            sujeitos_unicos = set()
            for s in subjects:
                if len(s) > 0:
                    sujeitos_unicos.add(s[0])
            print(f"Sujeitos (de lista aninhada): {len(sujeitos_unicos)}")
            print(f"Primeiros 10: {list(sujeitos_unicos)[:10]}")
        else:
            print(f"Sujeitos: {len(set(subjects))} unicos")
            print(f"Primeiros 10: {subjects[:10]}")
    elif hasattr(subjects, "shape"):
        unicos = np.unique(subjects)
        print(f"Sujeitos: {len(unicos)} unicos")
        print(f"Primeiros 10: {unicos[:10]}")

print()

# Numero de segmentos
if "Signals" in subset:
    sig_shape = subset["Signals"].shape
    print(f"Signals shape: {sig_shape}")
    print(f"Total de segmentos: {sig_shape[0]}")
    if len(sig_shape) >= 3:
        print(f"Canais: {sig_shape[1]}")
        print(f"Amostras por segmento: {sig_shape[2]}")

Sujeitos (de lista aninhada): 1293
Primeiros 10: ['p003675_1', 'p002328_1', 'p003699_1', 'p005416_1', 'p000548_1', 'p003489_1', 'p003332_1', 'p003873_1', 'p005145_1', 'p006021_1']

Signals shape: (51720, 3, 1250)
Total de segmentos: 51720
Canais: 3
Amostras por segmento: 1250


## 6. Resumo executivo

Com base na exploração acima, será possível decidir:

1. Se podemos usar **o mesmo loop do NB07** adaptado para CalBased
2. Qual estratégia de calibração aplicar (segmentos `Cal_*` separados ou split por flag)
3. Quantos sujeitos e ciclos esperados na extração completa


In [6]:
print(f"NB09a concluido em: {datetime.now().strftime('%H:%M:%S')}")
print()
print("PROXIMO PASSO: com base nos achados acima, gerar o NB09 completo.")

NB09a concluido em: 19:13:56

PROXIMO PASSO: com base nos achados acima, gerar o NB09 completo.
